# C11-neural-training — Practice p06 — Solution


**Type:** constrained coding · **Difficulty:** intro · **Concepts:** cross-entropy-loss


For each row, \(\log\sum_c e^{z_c}=m+\log\sum_c e^{z_c-m}\).
Subtracting the correct shifted logit from the shifted log-normalizer cancels
the maximum algebraically and never forms a possibly underflowed probability.


In [ ]:
import numpy as np

def stable_cross_entropy(logits, labels):
    values = np.asarray(logits, dtype=np.float64)
    targets = np.asarray(labels)
    if values.ndim != 2 or targets.ndim != 1:
        raise ValueError("expected logits (N,C) and labels (N,)")
    n, c = values.shape
    if n != targets.shape[0] or c == 0 or n == 0:
        raise ValueError("invalid batch or class dimensions")
    if not np.issubdtype(targets.dtype, np.integer):
        raise ValueError("labels must be integers")
    if np.any(targets < 0) or np.any(targets >= c):
        raise ValueError("label out of range")
    shifted = values - values.max(axis=1, keepdims=True)
    log_norm = np.log(np.exp(shifted).sum(axis=1))
    return float(np.mean(log_norm - shifted[np.arange(n), targets]))

probe_p06 = np.array([[10000.0, 9998.0], [-10000.0, -9999.0]])
labels_p06 = np.array([0, 1])
loss_p06 = stable_cross_entropy(probe_p06, labels_p06)


### Answer check


In [ ]:
expected_p06 = float(np.mean([np.log1p(np.exp(-2.0)), np.log1p(np.exp(-1.0))]))
assert isinstance(loss_p06, float) and np.isfinite(loss_p06)
assert np.isclose(loss_p06, expected_p06, atol=1e-12, rtol=1e-10)
assert np.isclose(loss_p06, stable_cross_entropy(probe_p06 + np.array([[71.0], [-83.0]]), labels_p06), atol=1e-12, rtol=1e-10)
for bad_logits, bad_labels in [(np.ones(2), labels_p06), (probe_p06, np.array([[0, 1]])), (probe_p06, np.array([0, 2]))]:
    try:
        stable_cross_entropy(bad_logits, bad_labels)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid input was accepted")
